# Wikipedia RAG Indexing Pipeline

Creates a vector index for RAG evaluation with popularity metadata.

## Steps
1. Load QA datasets from HuggingFace
2. Load Wikipedia corpus
3. Add popularity metadata
4. Create vector index

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from rag.native_rag_service import nativeRagService, IndexingConfig
from config import DATA_DIR, CACHE_DIR
from llm.openAi_service import OpenAIService
import numpy as np
from tqdm.auto import tqdm
import asyncio
import logging

# Suppress noisy HTTP logs from libraries
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

# ============================================================================
# CONFIGURATION
# ============================================================================

# Datasets
QA_DATASETS = []
WIKIPEDIA_DATASET = "facebook/kilt_wikipedia"
WIKIPEDIA_VERSION = "2019-08-01"
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"

# Indexing
NAME = "wiki_600k_balanced_qa_s_250_only"
COLLECTION_NAME = "native_rag"
N_RANDOM_SAMPLES = 600_000  # Set to None to use all data
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

# Paths - Updated to separate index and questions
COLLECTION_ROOT = Path(DATA_DIR) / NAME
COLLECTION_PATH = COLLECTION_ROOT / "index"  # Vector DB location
QUESTIONS_PATH = COLLECTION_ROOT / "train_questions.parquet"

# Processing
BATCH_SIZE = 1000
RESUME_FROM_ROW = 0
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Balancing
BALANCE_DECILES = True # Ensure equal questions across 10 popularity deciles
ADD_SYNTHETIC_QUESTIONS = True # Generate questions for under-represented deciles
MIN_QUESTIONS_PER_DECILE = 250 # Target number of questions per decile
MODEL_NAME = "gpt-4.1-nano"  # Model for synthetic question generation

# Async generation settings (NEW - for efficiency)
SYNTHETIC_BATCH_SIZE = 500  # How many questions to generate in parallel

print(f"✓ Config loaded: {QA_DATASETS} → {COLLECTION_NAME}")
print(f"  Output Root: {COLLECTION_ROOT}")
print(f"  Balance: {BALANCE_DECILES} (Synthetic: {ADD_SYNTHETIC_QUESTIONS}, Target: {MIN_QUESTIONS_PER_DECILE})")

/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Config loaded: [] → native_rag
  Output Root: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_200k_balanced_qa_s_250_only
  Balance: True (Synthetic: True, Target: 250)


In [2]:
# ============================================================================
# STEP 1: Load QA Datasets
# ============================================================================

print("Loading QA datasets...")
qa_dfs = []

if not QA_DATASETS:
    print("  No QA datasets specified. Proceeding without QA data.")
    qa_df = pd.DataFrame(columns=[
        "question_text",
        "answers",
        "wikipedia_id",
        "dataset",
        "popularity_avg",
        "popularity_rank",
        "decile",
        "is_synthetic",
        "wikipedia_title",
    ])
else:
    for ds_name in QA_DATASETS:
        ds = load_dataset("Cyro1/popularity-enriched-qa-datasets", ds_name, split="train+test", cache_dir=CACHE_DIR)
        df = ds.to_pandas()
        df["dataset"] = ds_name
        qa_dfs.append(df)
        print(f"  {ds_name}: {len(df):,} questions")

    qa_df = pd.concat(qa_dfs, ignore_index=True)

    # Normalize rank column naming (standard: popularity_rank)
    if "rank_avg" in qa_df.columns:
        if "popularity_rank" in qa_df.columns:
            qa_df["popularity_rank"] = qa_df["popularity_rank"].combine_first(qa_df["rank_avg"])
            qa_df = qa_df.drop(columns=["rank_avg"])
        else:
            qa_df = qa_df.rename(columns={"rank_avg": "popularity_rank"})

    # Clean IDs
    qa_df = qa_df.dropna(subset=["wikipedia_id"])
    qa_df["wikipedia_id"] = qa_df["wikipedia_id"].astype(int)

# Identify documents needed for the GROUND TRUTH questions
# We track this to ensure we load them from Wikipedia later, even if we balance/augment
required_doc_ids = set(qa_df["wikipedia_id"]) if "wikipedia_id" in qa_df.columns else set()

print(f"\n✓ Initial QA Set: {len(qa_df):,} questions")
print(f"✓ Unique docs needed: {len(required_doc_ids):,}")
print("  (Balancing will occur in Step 4 to allow synthetic generation from loaded text)")

Loading QA datasets...
  No QA datasets specified. Proceeding without QA data.

✓ Initial QA Set: 0 questions
✓ Unique docs needed: 0
  (Balancing will occur in Step 4 to allow synthetic generation from loaded text)


In [3]:
# ============================================================================
# STEP 2: Load Wikipedia
# ============================================================================

print("Loading Wikipedia...")
# Keep a reference to full dataset for retrieval of missing docs
full_wiki_ds = load_dataset(WIKIPEDIA_DATASET, WIKIPEDIA_VERSION, split="full", cache_dir=CACHE_DIR)
full_wiki_ds = full_wiki_ds.select_columns(["wikipedia_id", "wikipedia_title", "text"])

if N_RANDOM_SAMPLES is not None:
    print(f"Sampling {N_RANDOM_SAMPLES:,} random documents from Wikipedia...")
    wiki_ds = full_wiki_ds.shuffle(seed=42).select(range(N_RANDOM_SAMPLES))
else:
    wiki_ds = full_wiki_ds

print(f"✓ Loaded {len(wiki_ds):,} articles")

# OPTIMIZED: Use vectorized numpy operations instead of loop
print("Checking for missing documents...")
existing_ids = set(int(i) for i in wiki_ds["wikipedia_id"] if i is not None)

missing_ids = required_doc_ids - existing_ids
print(f"✓ Missing: {len(missing_ids):,}")

# Add missing docs if needed
if missing_ids:
    # OPTIMIZED: Use filter with batched processing for speed
    missing_ds = full_wiki_ds.filter(
        lambda x: int(x["wikipedia_id"]) in missing_ids, 
        desc="Finding missing",
        batched=False  # Keep False for set membership check
    )
    wiki_ds = concatenate_datasets([wiki_ds, missing_ds])
    print(f"✓ Added missing docs. Total: {len(wiki_ds):,}")


# Flatten KILT text structure - ALWAYS CHECK
print("Normalizing text format...")

def flatten_text(batch):
    return {
        "text": [
            "\n".join(t["paragraph"]) if isinstance(t, dict) and "paragraph" in t else str(t)
            for t in batch["text"]
        ]
    }

# Apply to first item to check if needed
needs_flattening = False
if len(wiki_ds) > 0:
    sample_text = wiki_ds[0]["text"]
    if isinstance(sample_text, dict):
        needs_flattening = True

if needs_flattening:
    wiki_ds = wiki_ds.map(flatten_text, batched=True, desc="Flattening text")
    print("✓ Text flattened")
else:
    print("✓ Text already flat (or empty)")

Loading Wikipedia...


Loading dataset shards:   0%|          | 0/59 [00:00<?, ?it/s]

Sampling 240,000 random documents from Wikipedia...
✓ Loaded 240,000 articles
Checking for missing documents...
✓ Missing: 0
Normalizing text format...
✓ Text flattened


In [4]:
# ============================================================================
# STEP 3: Add Popularity Metadata
# ============================================================================

print("Loading popularity data...")
pop_ds = load_dataset(POPULARITY_DATASET, split="train+test", cache_dir=CACHE_DIR)

# Detect columns
cols = pop_ds.column_names
id_col = "wikipedia_id" if "wikipedia_id" in cols else "id"
rank_col = next((c for c in ["rank_avg", "avg_rank"] if c in cols), None)

# MAJOR OPTIMIZATION: Get needed IDs early
needed_ids = set(int(x) for x in wiki_ds["wikipedia_id"] if x is not None)
print(f"✓ Target documents: {len(needed_ids):,}")

print("Processing popularity metadata (OPTIMIZED - Vectorized)...")

# STEP 1: Load MINIMAL data first to calculate global deciles
# We need all rows for correct global decile calculation, but only ID and popularity columns
print("  Loading popularity scores for global decile calculation...")
pop_df_minimal = pop_ds.select_columns([id_col, "popularity_avg"]).to_pandas()
pop_df_minimal[id_col] = pd.to_numeric(pop_df_minimal[id_col], errors='coerce').fillna(-1).astype(int)

# STEP 2: Calculate Global Deciles (Vectorized on full dataset)
print("  Calculating global deciles...")
pop_df_minimal["decile"] = pd.qcut(
    pop_df_minimal["popularity_avg"].rank(method="first"),
    10,
    labels=False,
)

# STEP 3: FILTER EARLY - Keep only needed rows BEFORE loading additional columns
print(f"  Filtering from {len(pop_df_minimal):,} to {len(needed_ids):,} rows...")
relevant_pop_df = pop_df_minimal[pop_df_minimal[id_col].isin(needed_ids)].copy()

# STEP 4: Now add rank column if needed (only for relevant rows)
if rank_col and rank_col in pop_ds.column_names:
    print("  Adding rank data for relevant documents only...")
    # Get rank data only for needed IDs - MUCH more efficient
    pop_ds_filtered = pop_ds.filter(
        lambda batch: [int(i) in needed_ids for i in batch[id_col]],
        batched=True,
        batch_size=10000,
        desc="Filtering popularity data",
    )
    rank_df = pop_ds_filtered.select_columns([id_col, rank_col]).to_pandas()
    rank_df[id_col] = rank_df[id_col].astype(int)

    # Merge rank data
    relevant_pop_df = relevant_pop_df.merge(rank_df, on=id_col, how="left")

# Standardize rank column name
if rank_col:
    relevant_pop_df = relevant_pop_df.rename(columns={rank_col: "popularity_rank"})
else:
    relevant_pop_df["popularity_rank"] = None

# STEP 5: Build SMALL lookup dictionary (only ~180k entries instead of 6M!)
print("  Building optimized lookup...")
pop_lookup = relevant_pop_df.set_index(id_col).to_dict(orient="index")

# Cleanup
del pop_df_minimal, pop_ds
if rank_col:
    del pop_ds_filtered, rank_df
import gc
gc.collect()

print(f"✓ Built lookup with {len(pop_lookup):,} entries")

# Merge with Wikipedia using batched map for efficiency
print("Merging metadata...")

def merge_batch(batch):
    """Vectorized metadata merge - MUCH faster than row-by-row"""
    ids = [int(i) for i in batch["wikipedia_id"]]
    defaults = {"popularity_avg": None, "popularity_rank": None, "decile": -1}
    meta_list = [pop_lookup.get(i, defaults) for i in ids]

    return {
        "popularity_avg": [m.get("popularity_avg") for m in meta_list],
        "popularity_rank": [m.get("popularity_rank") for m in meta_list],
        "decile": [m.get("decile", -1) for m in meta_list],
    }

wiki_ds_with_pop = wiki_ds.map(
    merge_batch,
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Merging",
)

# Identify which deciles we actually have in our index
present_deciles = set(np.unique(wiki_ds_with_pop["decile"]))
print(f"✓ Ready for indexing: {len(wiki_ds_with_pop):,} documents")

Loading popularity data...


✓ Target documents: 240,000
Processing popularity metadata (OPTIMIZED - Vectorized)...
  Loading popularity scores for global decile calculation...
  Calculating global deciles...
  Filtering from 5,903,530 to 240,000 rows...
  Adding rank data for relevant documents only...
  Building optimized lookup...
✓ Built lookup with 240,000 entries
Merging metadata...
✓ Ready for indexing: 240,000 documents


In [ ]:
# ============================================================================
# STEP 3.5: Balance and Augment QA (compact + efficient)
# ============================================================================

if not BALANCE_DECILES:
    print("Skipping balancing.")
else:
    print("\n⚖️ Balancing QA Dataset...")

    # Map questions to deciles + popularity (avoid merge overhead)
    decile_map = {doc_id: meta.get("decile", -1) for doc_id, meta in pop_lookup.items()}
    pop_map = {doc_id: meta.get("popularity_avg") for doc_id, meta in pop_lookup.items()}
    rank_map = {doc_id: meta.get("popularity_rank") for doc_id, meta in pop_lookup.items()}

    qa_df["decile"] = qa_df["wikipedia_id"].map(decile_map)

    if "popularity_avg" in qa_df.columns:
        qa_df["popularity_avg"] = qa_df["popularity_avg"].combine_first(qa_df["wikipedia_id"].map(pop_map))
    else:
        qa_df["popularity_avg"] = qa_df["wikipedia_id"].map(pop_map)

    if "popularity_rank" in qa_df.columns:
        qa_df["popularity_rank"] = qa_df["popularity_rank"].combine_first(qa_df["wikipedia_id"].map(rank_map))
    else:
        qa_df["popularity_rank"] = qa_df["wikipedia_id"].map(rank_map)

    qa_df["decile"] = qa_df["decile"].fillna(-1).astype(int)
    qa_df["is_synthetic"] = False

    # Drop unknown decile questions
    invalid = (qa_df["decile"] == -1).sum()
    if invalid:
        print(f"  Warning: Dropped {invalid} questions with unknown decile")
    qa_df = qa_df[qa_df["decile"] != -1].copy()

    counts = qa_df["decile"].value_counts().sort_index()
    print(f"  Current Distribution:\n{counts}")

    target_count = MIN_QUESTIONS_PER_DECILE if ADD_SYNTHETIC_QUESTIONS else counts.min()
    print(
        f"  Target per decile: {target_count} ("
        f"{'Synthetic enabled' if ADD_SYNTHETIC_QUESTIONS else 'Limited by smallest decile'})"
    )

    # Init LLM if needed
    llm_service = None
    if ADD_SYNTHETIC_QUESTIONS:
        try:
            llm_service = OpenAIService(temperature=0.7, request_timeout=None, model_name=MODEL_NAME)
            print("  ✓ LLM Service Initialized (rate-limited)")
        except Exception as e:
            print(f"  ❌ Failed to init LLM: {e}")
            print("  Disabling synthetic generation.")
            ADD_SYNTHETIC_QUESTIONS = False
            target_count = counts.min()

    if ADD_SYNTHETIC_QUESTIONS:
        # Precompute decile indices once (faster than repeated dataset filters)
        wiki_deciles = np.asarray(wiki_ds_with_pop["decile"])
        decile_to_indices = {decile: np.where(wiki_deciles == decile)[0] for decile in range(10)}

        async def _generate_questions_for_decile(decile: int, needed: int):
            indices = decile_to_indices.get(decile)
            if indices is None or len(indices) == 0:
                return []

            idxs = np.random.choice(indices, size=needed, replace=(needed > len(indices)))
            batch_size = min(SYNTHETIC_BATCH_SIZE, max(1, needed))

            async def _gen_one(idx):
                doc = wiki_ds_with_pop[int(idx)]
                text = doc["text"]
                title = doc["wikipedia_title"]
                doc_id = int(doc["wikipedia_id"])

                prompt = (
                    f"Generate a single factual question that can be answered by the following text from Wikipedia article '{title}'.\n"
                    f"Text: {text[:2000]}\n"
                    f"Question:"
                )

                try:
                    response = await llm_service.ainvoke(prompt)
                    question = response.strip()
                    if not question:
                        return None
                    return {
                        "question_text": question,
                        "answer_texts": [],
                        "wikipedia_id": doc_id,
                        "wikipedia_title": doc.get("wikipedia_title"),
                        "dataset": "synthetic",
                        "decile": decile,
                        "is_synthetic": True,
                        "popularity_avg": doc.get("popularity_avg"),
                        "popularity_rank": doc.get("popularity_rank"),
                    }
                except Exception:
                    return None
ch

            tasks = [_gen_one(i) for i in idxs]
            results = []
            for start in range(0, len(tasks), batch_size):
                batch = tasks[start : start + batch_size]
                for coro in tqdm(asyncio.as_completed(batch), total=len(batch), desc=f"Decile {decile}"):
                    result = await coro
                    if result:
                        results.append(result)
            return results

        def _run_async(coro):
            try:
                loop = asyncio.get_running_loop()
            except RuntimeError:
                return asyncio.run(coro)
            else:
                import nest_asyncio
                nest_asyncio.apply()
                return loop.run_until_complete(coro)

    final_dfs = []
    for decile in range(10):
        current_df = qa_df[qa_df["decile"] == decile]
        curr_count = len(current_df)

        if curr_count >= target_count:
            final_dfs.append(current_df.sample(n=target_count, random_state=42))
            continue

        final_dfs.append(current_df)

        if ADD_SYNTHETIC_QUESTIONS:
            needed = target_count - curr_count
            print(f"  Decile {decile}: Generating {needed} synthetic questions...")

            new_qs = _run_async(_generate_questions_for_decile(decile, needed))

            if new_qs:
                final_dfs.append(pd.DataFrame(new_qs))
                print(f"    ✓ Generated {len(new_qs)} questions")
            if len(new_qs) < needed:
                print(f"    ⚠️ Shortfall: {needed - len(new_qs)} questions (likely timeouts/errors)")

    qa_df = pd.concat(final_dfs, ignore_index=True)

    # Ensure consistent schema and types
    if "question_text" not in qa_df.columns and "question" in qa_df.columns:
        qa_df = qa_df.rename(columns={"question": "question_text"})

    qa_df["wikipedia_id"] = qa_df["wikipedia_id"].astype(int)

    print(f"✓ Balanced Total: {len(qa_df):,} questions")
    print(f"New Distribution:\n{qa_df['decile'].value_counts().sort_index()}")


⚖️ Balancing QA Dataset...
  Current Distribution:
Series([], Name: count, dtype: int64)
  Target per decile: 250 (Synthetic enabled)


Initialized OpenAI service with model: gpt-4.1-nano


  ✓ LLM Service Initialized (rate-limited)
  Decile 0: Generating 250 synthetic questions...


Decile 0:   0%|          | 0/250 [00:00<?, ?it/s]

    ✓ Generated 250 questions
  Decile 1: Generating 250 synthetic questions...


...

Decile 8:   0%|          | 0/250 [00:00<?, ?it/s]

    ✓ Generated 250 questions
  Decile 9: Generating 250 synthetic questions...


Decile 9:   0%|          | 0/250 [00:00<?, ?it/s]

    ✓ Generated 250 questions
✓ Balanced Total: 2,500 questions
New Distribution:
decile
0    250
1    250
2    250
3    250
4    250
5    250
6    250
7    250
8    250
9    250
Name: count, dtype: int64


/var/folders/14/7y09r8b90rd3t_63ryrt3b1r0000gn/T/ipykernel_6484/3646194236.py:144: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  qa_df = pd.concat(final_dfs, ignore_index=True)


In [6]:
# ============================================================================
# STEP 4: Create Index
# ============================================================================

print("Creating index...")
service = nativeRagService(IndexingConfig(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    embedding_provider="huggingface",
    embedding_model=EMBEDDING_MODEL,
    use_progress=True
))

# Ensure output directory exists
COLLECTION_ROOT.mkdir(parents=True, exist_ok=True)

# Save the questions used for this run
print(f"Saving training questions to {QUESTIONS_PATH}...")
qa_df.to_parquet(QUESTIONS_PATH)

index, num_chunks = service.index_from_dataset(
    ds=wiki_ds_with_pop,
    text_field="text",
    metadata_fields=["wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank"],
    output_dir=COLLECTION_PATH,
    collection_name=COLLECTION_NAME,
    progress_bar=True,
    batch_size=BATCH_SIZE,
    resume_from_row=RESUME_FROM_ROW
)

print(f"\n✅ DONE")
print(f"  Documents: {len(wiki_ds_with_pop):,}")
print(f"  Chunks: {num_chunks:,}")
print(f"  Index: {COLLECTION_PATH}")
print(f"  Questions: {QUESTIONS_PATH}")
print(f"\nNext: Run rag_evaluation.ipynb")

Creating index...


Use pytorch device_name: mps
Load pretrained SentenceTransformer: intfloat/multilingual-e5-small
Processing dataset in batches of 1000 rows...
Deleting existing index at /Users/cyro/Documents/VSC/PopularityBias/data/wiki_200k_balanced_qa_s_250_only/index/chroma


Saving training questions to /Users/cyro/Documents/VSC/PopularityBias/data/wiki_200k_balanced_qa_s_250_only/train_questions.parquet...


100%|██████████| 240000/240000 [5:55:27<00:00, 11.25rows/s]  


✅ DONE
  Documents: 240,000
  Chunks: 1,709,721
  Index: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_200k_balanced_qa_s_250_only/index
  Questions: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_200k_balanced_qa_s_250_only/train_questions.parquet

Next: Run rag_evaluation.ipynb
